# Phase 0 — 00 · Setup on the IDA GPU pod

Builds a dedicated conda env for the pipeline, installs DIAMOND, registers a Jupyter kernel,
and verifies every tool, repo file and database.

**Run the first two cells with the default `base` kernel.** After the bootstrap registers the
new kernel, switch to **Python 3.12 (geneweaver)** and carry on from section 4.

Three things about this pod that shape the setup:

| | |
|---|---|
| **You are root** (`scc: anyuid`) | conda and apt both work; no user-prefix tricks needed |
| **Only `/mnt/primary` has space** | 164 TB free; `/mnt/indices` and `/mnt/resources` are **100 % full**, `datasets` and `llm` 95 % |
| **The pod times out after 2 h** and is killed if CPU drops below 20 % | the DIAMOND stage keeps CPU busy, but don't leave the pod idle mid-run |

The pipeline env is kept **separate from the Bakta env on purpose** — Bakta depends on diamond,
pyrodigal, pyhmmer and biopython itself and pins its own versions, which would silently change the
tool versions Phase 0's baseline was measured with.

## 1 · Get the repository onto /mnt/primary

Skip if it is already cloned.

In [ ]:
import subprocess, os
from pathlib import Path

PRIMARY = Path("/mnt/primary")
REPO = PRIMARY / "Geneweaver_v3"

if not REPO.is_dir():
    print("Clone it first, e.g.:")
    print(f"  cd {PRIMARY} && git clone <your-remote> Geneweaver_v3")
    print("  cd Geneweaver_v3 && git merge dev        # the four annotator scripts live on dev")
else:
    print("repo:", REPO)
    print(subprocess.run(["git","-C",str(REPO),"log","--oneline","-1"],
                         capture_output=True, text=True).stdout.strip())
    for f in ["homology_annotator.py","pfam_annotator.py","go_mapper.py","annotation_combiner.py"]:
        p = REPO / "Structural_annotation" / f
        print(("  ok   " if p.is_file() else "  MISSING (merge dev) "), f)

## 2 · Bootstrap the environment

Creates `/mnt/primary/envs/geneweaver` with Python 3.12, the pinned packages and DIAMOND 2.2.8,
then registers the kernel. Takes a few minutes. Idempotent.

In [ ]:
CLUSTER = REPO / "phase0-bundle" / "cluster"
r = subprocess.run(["bash", str(CLUSTER / "bootstrap.sh")], capture_output=True, text=True)
print(r.stdout[-5000:]); print(r.stderr[-2000:])
assert r.returncode == 0, "bootstrap failed"

---
### Switch kernel now

**Kernel → Change Kernel → Python 3.12 (geneweaver)**, then continue below.

## 3 · Confirm you are on the right kernel

In [ ]:
import sys
print(sys.version)
assert sys.version_info[:2] == (3, 12), "wrong kernel - switch to Python 3.12 (geneweaver)"
import pyrodigal, pyhmmer, Bio
print("pyrodigal", pyrodigal.__version__, "| pyhmmer", pyhmmer.__version__, "| biopython", Bio.__version__)

## 4 · Paths

Creates `cluster/paths.json` pre-filled for this pod. **Check `databases` and `bakta_db` point
where you actually put them**, then re-run.

In [ ]:
import json
from pathlib import Path
PRIMARY = Path("/mnt/primary"); REPO = PRIMARY / "Geneweaver_v3"
CLUSTER = REPO / "phase0-bundle" / "cluster"
paths_file = CLUSTER / "paths.json"
if not paths_file.exists():
    cfg = json.loads((CLUSTER / "paths.example.json").read_text())
    cfg.pop("_comment", None); cfg["repo"] = str(REPO)
    paths_file.write_text(json.dumps(cfg, indent=2) + "\n")
    print("created", paths_file, "- check the paths, then re-run this cell\n")
print(paths_file.read_text())

## 5 · Reference databases

~2.8 GB, one-off. Skip if you already copied them onto `/mnt/primary`.

The no-*E. coli* Swiss-Prot copy is **derived, not downloaded** — the script filters
*Escherichia*/*Shigella* and asserts the result is 545,523 entries, exactly matching the
database the published baseline used.

In [ ]:
import subprocess, json
cfg = json.loads(paths_file.read_text())
env = {**os.environ, "GENEWEAVER_DB": cfg["databases"]}
r = subprocess.run(["bash", str(CLUSTER.parent / "scripts" / "fetch_databases.sh")],
                   env=env, capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-3000:])

## 6 · Verify

Fast check first; then the md5 pass once, to confirm the databases are byte-identical to the baseline.

In [ ]:
r = subprocess.run([sys.executable, str(CLUSTER / "check_env.py")], capture_output=True, text=True)
print(r.stdout); print(r.stderr, file=sys.stderr)
assert r.returncode == 0, "environment check failed - fix the problems above"

In [ ]:
# Slow (~2.8 GB). Run once, then comment out.
r = subprocess.run([sys.executable, str(CLUSTER / "check_env.py"), "--verify-md5"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr, file=sys.stderr)

---
Ready. Continue in **`01_phase0.ipynb`** (same kernel).